---

# **Introducción**

El **HydroSOS** es una iniciativa de la Organización Meteorológica Mundial (OMM) cuyo objetivo es aumentar la resistencia mundial a los riesgos hidroclimáticos y la gestión de los recursos hídricos mediante la elaboración de evaluaciones del estado y las perspectivas hidrológicas a diferentes escalas en todo el mundo. La idea es permitir la evaluación del estado actual de los recursos hídricos respecto a las condiciones "esperadas" o "normales", así como perspectivas hidrológicas de cómo este estado puede cambiar durante las siguientes semanas y meses. En el contexto de la iniciativa HydroSOS se ha creado un portal de demonstraciones de estado y perspectivas futuras, al que se puede acceder en: https://wmohydrosos.ceh.ac.uk/portal.

En este tutorial, utilizaremos un script para generar **perspectivas hidrológicas basadas en HydroSOS** en puntos donde hay pronósticos de caudales. La metodología consiste en transformar los caudales pronosticados en un percentil relativo al histórico de la correspondiente época del año, por ejemplo, del mes en curso, y clasificar el percentil resultante según una categoría de flujo predeterminada.


---

# **Clonando el repositório del HydroSOS**

Clonaremos el repositório del WMO HydroSOS hospedado en Github para para tener acceso a los scripts de cálculo de los percentiles y también a un ejemplo de resultados de previsión de caudales.

*   Obs: Para utilizar el comando `git clone` en alguna otra plataforma de ejecución de notebooks/scripts python (e.g., ***Jupyter***, ***Visual Studio Code***), hay que certificarse de que el `Git` se encuentra instalado en su ordenador. En caso contrário, el git se puede descargar en (https://git-scm.com/downloads)

In [ ]:
!git clone https://github.com/wmo-im/HydroSOS/

---

# **Importando librerías python**

Importaremos las librerías necesarias para la realización del tutorial:

In [ ]:
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

Crearemos automaticamente una carpeta output en el diretorio raiz donde vamos exportar los resultados:

In [ ]:
# Create the folder if it doesn't exist
os.makedirs('output_hydrosos', exist_ok=True)

---
# **1. Aplicando la metodología HydroSOS con el script `forecastcalc.py`**

1.1. Esta sección muestra cómo generar las perspectivas hidrológicas para el HydroSOS utilizando el script `forecastcalc.py`, disponible en el repositorio de Github (https://github.com/wmo-im/HydroSOS).

* El script calculará las mismas categorías que las de `statuscalc.py`, basandose en los datos históricos de caudal del periodo de referencia (por ejemplo, 1991-2020). Nota: **si las previsiones que se van a clasificar han sido generadas por un modelo hidrológico conceptual lluvia-escorrentía, los datos históricos utilizados aquí deberían ser los modelizados**, es decir, obtenidos a partir de una simulación histórica con el mismo modelo. Esto reduce la influencia de los errores del modelo (en relación con la media y la variabilidad de la serie) en el cálculo de los percentiles, mejorando la coherencia estadística entre los periodos futuro e histórico.

* Cada previsión o miembro del conjunto tendrá su percentil calculado en relación con la serie de datos históricos observados o simulados, según la metodología de previsión adoptada.

1.2. El script procesará los pares de datos observado/pronosticado **para cada estación disponible**, y debe ejecutarse con los siguientes parámetros:

`python forecastcalc.py obs_dir forecast_dir output_dir --obsDirStartingMonth --varName`


Donde:

* `obs_dir`: directorio que contiene únicamente archivos .csv de datos históricos de caudales (observados o simulados), como **series temporales diarias**. Los nombres de los archivos deben tener el formato **X_CatchmentID.csv**, donde "X" es un prefijo y "CatchmentID" es el codigo de la estación hidrométrica. Consulte los archivos en la carpeta HydroSOS/example_data/forecast/input/obs_dir/ para ver el formato esperado de estos archivos;

* `forecast_dir`: directorio que contiene únicamente archivos .csv de  **previsiones con paso de tiempo mensual**, donde cada miembro del conjunto (ensemble) debe estar separado en un solo archivo .csv. Los nombres de los archivos deben tener el formato **X_ENS_CatchmentID.csv**, donde "ENS" es una referéncia al miembro (por ejemplo, A, B, C, ... o M1, M2, M3, etc.). Consulte los archivos en la carpeta HydroSOS/example_data/forecast/input/forecast_dir/ para ver el formato esperado de estos archivos;

* `output_dir`: directorio al que se envían los archivos procesados con la metodología HydroSOS;

* `obsDirStartingMonth`: mes inicial de los datos existentes en la carpeta `obs_dir` (por defecto enero = 1).

* `varName`: Nombre de la variable en los datos (por defecto = Discharge)


**Importante**: El periodo de referencia será determinado por la disponibilidad de datos históricos de caudal diario. Por ejemplo, si la serie de datos en la carpeta `obs_dir` comienza el 01/01/1991 y termina el 31/12/2020, el periodo para calcular las bandas de percentiles para clasificación de los pronósticos estará comprendido entre 1991-2020.

Para más detalles consulte la información disponible en `HydroSOS/README.md`.

In [ ]:
# Run the script using shell command
!python ./HydroSOS/forecast/forecastcalc.py ./HydroSOS/example_data/forecast/input/obs_dir/ ./HydroSOS/example_data/forecast/input/forecast_dir/ ./output_hydrosos/


---
1.3. Si el script se ejecuta correctamente, deberían generarse algunas carpetas en la salida "output" que contengan archivos ".csv". Estas carpetas deberían incluir:

* *accumulated* : producto de previsión agregado (promedio de los meses anteriores hasta la fecha actual)
* *single* : producto de previsión con paso de tiempo mensual
* *status* : datos históricos utilizados para calcular percentiles y generar las categorías.

---
# **2. Cargando las tablas con los resultados de HydroSOS**

2.1. Una vez generadas las tablas ejecutando el script `forecastcalc.py`, vamos a establecer algunos parámetros para leer y plotar los resultados:

* `station_id`: código de la estación hidrométrica para la lectura de los datos;
* `forecast_type`: agregación del pronóstico que se va a graficar ("single" o "accumulated");
* `color_codes_fcst`: códigos de color para representar las bandas categoricas en el pronóstico.


In [ ]:
# Select station to plot (CatchmentID)
station_ID = '1'

# Folder where files were saved
folder_output = 'output_hydrosos'

# Select forecast aggregation time
# Type: 'accumulated' (e.g., X-month averaged) or 'single' (for monthly lead time)
forecast_type = 'single'

# Color codes (HEX) according to HydroSOS standards (5 categories)
color_codes_fcst = ['#CD233f','#FFA885','#E7E2BC','#8ECEEE','#2C7DCD']

2.2. Cargando salidas de información histórica (status):

In [ ]:
# Time series (obsSim)
df_status = pd.read_csv(f'{folder_output}/status/status/{station_ID}_status.csv')
# Percentile bands
df_status_bands = pd.read_csv(f'{folder_output}/status/statusBands/{station_ID}_bands.csv')

print('Datos historicos:')
display(df_status)
print('Bandas calculadas con información historica:')
display(df_status_bands)

2.3. Cargando los resultados de las perspectivas hidrológicas futuras:

In [ ]:
# Reading the output files from script forecastcalc.py
df_forecasts = pd.read_csv(f'{folder_output}/{forecast_type}/forecasts/{station_ID}_forecasts.csv')
df_counts = pd.read_csv(f'{folder_output}/{forecast_type}/counts/{station_ID}_counts.csv')
df_fcst_bands = pd.read_csv(f'{folder_output}/{forecast_type}/forecastBands/{station_ID}_bands.csv')
df_percentiles = pd.read_csv(f'{folder_output}/{forecast_type}/percentiles/{station_ID}_percentiles.csv')

# Setting indices to dataframes
df_forecasts.set_index('date', inplace=True)
df_counts.set_index('date',inplace=True)
df_fcst_bands.set_index('relative_month', inplace=True)

A continuación, trabajaremos con estas tablas para visualizar algunos resultados.

---
# **3. Verificando los gráficos de perspectivas hidrológicas**

## 3.1. Gráficos de previsión (trazos del ensemble)

La tabla  en la carpeta "**forecasts**" contiene las previsiones de caudal de cada miembro, con paso de tiempo mensual:

In [ ]:
display(df_forecasts)

In [ ]:
#@title Plotando los gráficos:
df_forecasts.plot(figsize=(12, 6),
                  legend=False,
                  color='gray',
                  ylabel='Caudal (m³/s)',
                  xlabel='Fecha de validez del pronóstico',
                  title=f'Trazos del ensemble ({forecast_type})')

## 3.2. Número de miembros en cada categoría (counts)

La tabla generada en la carpeta "**counts**" contiene el número de miembros del conjunto dentro de cada categoría de flujo predeterminada:

In [ ]:
display(df_counts)

Podemos obtener el tamaño del ensemble para calcular el % de miembros que entran en cada una de las categorías:

In [ ]:
# Calculating the percentage of members in each category
n_members = df_counts.sum(axis=1).values[0]
df_counts_prob = 100 * (df_counts / n_members)

display(df_counts_prob)

In [ ]:
#@title Plotando las probabilidades de pronóstico en gráficos de barras
fig, ax = plt.subplots(figsize=(8, 6))

ax = df_counts_prob.plot(ax=ax, kind='bar', ylabel='Probabilidad (%)',
                         xlabel='Fecha de validez del pronóstico', edgecolor='k',
                         color=color_codes_fcst, width=0.8, stacked='True')

# Set date labels to x axis
ax.set_xticklabels(df_counts_prob.index, rotation=45)

# Add the legend above the plot
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.set_title(f'% de miembros en cada categoría ({forecast_type})')

plt.tight_layout()

## 3.3. Bandas de categoría de previsión (forecast bands)

La tabla generada en la carpeta "**ForecastBands**" contiene los percentiles (y otras estadísticas) calculados con los datos históricos diarios. Estos percentiles son relativos al mes de la primera fecha que consta en nuestro pronóstico (`relative_month = 1`):

In [ ]:
df_fcst_bands

Separemos las estadísticas de la tabla para graficar los resultados:

In [ ]:
# Get only minimum and maximum values of the reference period, in the corresponding months in the forecast period
df_fcst_bands_min_max = df_fcst_bands[['min','max']]

# Get only percentiles of the reference period (i.e., by excluding other statistics)
df_fcst_bands_prct = df_fcst_bands.drop(columns=['min','mean','max'])

In [ ]:
# @title Plotando las bandas de percentil
fig,ax = plt.subplots(figsize=(8, 6))

##################################################3
# First part: Ploting the percentile bands

x = df_fcst_bands_prct.index  # Relative months starting from the forecast issue date
percentile_cols = df_fcst_bands_prct.columns # Names of the percentile columns (e.g., '10%', '25%', etc.)
n_bands = len(percentile_cols) - 1 # Number of bands between percentiles

# Band below the lowest climatological threshold (from 0 to first percentile)
ax.fill_between(x, 0, df_fcst_bands_prct.iloc[:, 0],
                 color=color_codes_fcst[0], alpha=0.7, label=f'<{percentile_cols[0]}'
)

# Bands between climatological percentiles
for i in range(n_bands):
    ax.fill_between(x, df_fcst_bands_prct.iloc[:, i], df_fcst_bands_prct.iloc[:, i + 1],
        color=color_codes_fcst[i + 1], label=f'{percentile_cols[i]}-{percentile_cols[i+1]}'
    )

# Band above the highest climatological threshold (from last percentile to max)
ax.fill_between(x, df_fcst_bands_prct.iloc[:, -1], df_fcst_bands_min_max['max'],
    color=color_codes_fcst[n_bands + 1], label=f'>{percentile_cols[-1]}'
)

# Plotting minimum and maximum historical (simulated) values
ax.plot(df_fcst_bands_min_max['min'], linestyle='--', color='silver', label='mínimo histórico')
ax.plot(df_fcst_bands_min_max['max'], linestyle='--', color='black', label='máximo histórico')


##################################################
# Second part: Ploting boxplots for the forecast ensemble

# Numbers to match the x-axis of the boxplot with the dates
positions = np.arange(1,len(df_forecasts.index) + 1)

ax.boxplot(
    [df_forecasts.loc[date] for date in df_forecasts.index],
     positions=positions,
     widths=0.6,
     patch_artist=True,
     boxprops=dict(facecolor='grey', color='black', alpha=0.3),
     medianprops=dict(color='black'),
     whiskerprops=dict(color='black'),
     capprops=dict(color='black'),
     flierprops=dict(markerfacecolor='black', marker='o', markersize=3, alpha=0.5)
 )

ax.set_xticklabels(df_forecasts.index, rotation=45)

plt.xlabel('Fecha de validez del pronóstico')
plt.ylabel('Caudal (m³/s)')
plt.title(f'Pronóstico con ESP ({forecast_type})')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.xlim(0.5, len(df_forecasts.index) + 0.5)
plt.tight_layout()
plt.show()

Finalmente, la tabla generada en la carpeta "percentiles" contiene estadísticas de nuestro ensemble:

In [ ]:
display(df_percentiles)

---
# **4. Ejercício para testar conocimientos**



Ahora usted será responsable de generar un producto de perspectiva hidrológica HydroSOS. Las instrucciones para ello son las siguientes:

**1. Generando el pronóstico**

* Utilice el **Cuaderno 4 - Ensemble Streamflow Prediction (ESP)** para generar una previsión ESP con início en **2025-01-01** y alcance de 6 meses (**180 días**).

**2. Guardando los archivos .csv**

* Guardar los resultados de la previsión en archivos .csv **individuales para cada miembro**. Los nombres deben ser compatibles con los datos de ejemplo de *forecast_dir* de la rutina `forecastcalc.py`.

* Guardar el archivo .csv de los **caudales históricos simulados** por el modelo hidrológico, filtrados para el periodo entre **1991-01-01 y 2020-12-31** (estos serán utilizados en lugar de los datos observados). El nombre del archivo debe ser compatible con los datos de ejemplo de *obs_dir* de la rutina `forecastcalc.py`.

**Sugerencia**: Utilice el comando de *shell* para comprimir los archivos .csv del ESP en un archivo .zip para facilitar su manejo en Colab.

**3. Generando las tablas HydroSOS con las perspectivas hidrológicas**

* Utilizando este cuaderno, ejecutar la rutina `forecastcalc.py` con los archivos generados en el paso 2.

**4. Graficando el resultado**

* Generar el gráfico de barras con las probabilidades del ESP (agregación single).

* Enviar el archivo (figura) con el resultado al enlace indicado (Sección **4 - Metodología HydroSOS -> Envío de la tarea**).

Nota: Si por alguna razón el gráfico no se ha podido generar correctamente, también se puede enviar la tabla con las probabilidades calculadas en cada categoría (df_counts_prob).
